# Hansen Ch.13 GMM — 计算

理论全文见 `Hansen_Ch13_Exercises_Solutions.md`（**13.1–13.28**）。

**13.27 AJR**、**13.28 Card** 两步有效 GMM 与 $J$ 统计量。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

def tsls(y, X, Z):
    PZ = Z @ inv(Z.T @ Z) @ Z.T
    b = inv(X.T @ PZ @ X) @ (X.T @ PZ @ y)
    return b, y - X @ b

def egmm_twostep(y, X, Z):
    """Linear IV efficient two-step GMM; returns beta, V_beta_hat, J, df."""
    n = len(y)
    b1, e1 = tsls(y, X, Z)
    Om = (Z * e1[:, None]).T @ (Z * e1[:, None]) / n
    W = inv(Om)
    ZX, ZY = Z.T @ X, Z.T @ y
    b = inv(ZX.T @ W @ ZX) @ (ZX.T @ W @ ZY)
    e = y - X @ b
    Om2 = (Z * e[:, None]).T @ (Z * e[:, None]) / n
    # Avar of sqrt(n)(b-beta) = (Q'Om^{-1}Q)^{-1}
    # Var(b) = that / n
    G = ZX / n  # approx Q'
    Asy = inv(G.T @ inv(Om2) @ G)  # avar of sqrt(n) beta
    V = Asy / n
    g = Z.T @ e / n
    J = float(n * g.T @ inv(Om2) @ g)
    df = Z.shape[1] - X.shape[1]
    return b, V, J, df


## Exercise 13.27 AJR：logmort + logmort² 工具

In [ ]:

ajr = pd.read_excel(ROOT / "AJR2001/AJR2001.xlsx")
d = ajr[["loggdp", "risk", "logmort0"]].dropna()
y = d.loggdp.values
X = np.column_stack([d.risk.values, np.ones(len(d))])
lm = d.logmort0.values
Z = np.column_stack([lm, lm**2, np.ones(len(d))])
b2, e2 = tsls(y, X, Z)
bg, Vg, J, df = egmm_twostep(y, X, Z)
print("n =", len(d))
print("2SLS: ", b2, "SE", np.sqrt(np.diag(inv(X.T @ (Z@inv(Z.T@Z)@Z.T) @ X) @ 
      ((Z@inv(Z.T@Z)@Z.T@X)*(e2[:,None])).T @ ((Z@inv(Z.T@Z)@Z.T@X)*(e2[:,None])) @ inv(X.T@(Z@inv(Z.T@Z)@Z.T)@X))))
print("EGMM: ", bg, "SE", np.sqrt(np.diag(Vg)))
print(f"J = {J:.4f}, df = {df}, p = {1-stats.chi2.cdf(J, df):.4f}")


## Exercise 13.28 Card：nearc4a, nearc4b 工具

In [ ]:

card = pd.read_excel(ROOT / "Card1995/Card1995.xlsx")
card["exper"] = card["age76"] - card["ed76"] - 6
card["exp2"] = (card["exper"] ** 2) / 100
cols = ["lwage76", "ed76", "exper", "exp2", "black", "smsa76r", "reg76r", "nearc4a", "nearc4b"]
d = card[cols].apply(pd.to_numeric, errors="coerce").dropna()
y = d.lwage76.values
Xexo = np.column_stack([d.exper, d.exp2, d.black, d.smsa76r, d.reg76r, np.ones(len(d))])
X = np.column_stack([d.ed76.values, Xexo])
Z = np.column_stack([d.nearc4a.values, d.nearc4b.values, Xexo])
b2, e2 = tsls(y, X, Z)
bg, Vg, J, df = egmm_twostep(y, X, Z)
print("n =", len(d))
print("2SLS edu =", b2[0])
print("EGMM edu =", bg[0], "SE =", np.sqrt(Vg[0, 0]))
print(f"J = {J:.4f}, df = {df}, p = {1-stats.chi2.cdf(J, df):.4f}")
